# AICE Professional 주피터라이트 실전 모의고사 16회 — 군 PBL 수행평가 재구성(탄약 수송 지연 · 증명서 판정 · 급식 식판 잔반)

## [ 필독 ] AICE Professional 모의고사 유의사항

- 공식 샘플문항(가공식품 카테고리 분류)과 **같은 시험지 형식**으로 만든 모의고사입니다. 3문항, 180분 기준입니다.
- 공식 시험과 같은 **Tabular / Text / Image 3유형**입니다. 2번 Text 문항은 공식 샘플(상품명 → 카테고리)과 같은 텍스트 분류입니다.
- 본인 스스로 모델을 만들고 **달성 목표 이상의 성능**이 나오도록 개발해야 합니다. 정답(테스트 라벨)은 제공되지 않습니다.
  대신 이 노트북 **맨 마지막 셀**(전체 자동 채점, 정답 키 인코딩)로 제출 파일을 스스로 채점할 수 있습니다(연습용).

## 사용할 수 있는 라이브러리 (주피터라이트 판)

이 모의고사는 **브라우저 주피터라이트(Pyodide · Python 3.14)** 에서 설치 없이 끝까지 풀 수 있도록 라이브러리를 제한했습니다.
필요한 라이브러리는 주피터라이트에 **미리 설치**되어 있습니다. 맨 위 '브라우저 실습' 셀은 확인만 하고 바로 끝납니다(Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다).

| 구분 | 쓸 수 있음 | 쓰지 않음(브라우저에서 안 됨) |
|---|---|---|
| 데이터·그림 | numpy · pandas · matplotlib · seaborn · Pillow | — |
| 머신러닝 | scikit-learn · LightGBM · XGBoost · imbalanced-learn · joblib | CatBoost |
| 텍스트 | scikit-learn 의 CountVectorizer · TfidfVectorizer · 내장 영어 불용어, **NLTK**(불용어·토큰화·어간 추출, 데이터는 `/drive/nltk_data` 에 미리 들어 있음) | 형태소 분석기 · BERT 등 사전학습 언어 모델 |
| 딥러닝 | `tensorflow.keras` 문법(TensorFlow.js 로 실행되는 대체 구현체, 미리 설치): `Sequential`·함수형(`Input` → 층 → `Model`) · Dense · Dropout · BatchNormalization · Conv2D · MaxPooling2D · GlobalAveragePooling2D · Embedding · LSTM · GRU · 콜백 3종(EarlyStopping · ModelCheckpoint · ReduceLROnPlateau) · `load_img` · `img_to_array` · `image_dataset_from_directory` · `ImageDataGenerator` · `model.save` / `load_model` · **MobileNet v1 전이학습**(`alpha=0.25`, 224×224, `weights='imagenet'` 만) | PyTorch · 다른 사전학습 모델(MobileNetV2/V3 · ResNet · EfficientNet 등) · 사용자 정의 층·손실 · GPU |

- 이미지는 64×64 정도로 줄여 작은 CNN 을 직접 학습하거나, MobileNet v1(alpha 0.25)로 특징을 뽑아 분류기를 학습하세요(전이학습). 브라우저 TensorFlow.js 는 CPython TensorFlow 와 수치가 조금 달라 딥러닝 점수가 약간 다르게 나올 수 있습니다.
- 학습이 오래 걸리면 epoch 를 줄이거나 조기 종료(EarlyStopping)를 쓰세요. 최신 Chrome·Edge 를 권장합니다.

## 시간 배분(180분)

| 문항 | 권장 시간 | 먼저 할 일 |
|---|---|---|
| 1. Tabular (30점) | 50분 | `info()`·`describe()` 로 자료형·결측 확인 → 기본 모델로 제출 파일부터 만들기 |
| 2. Text (35점) | 60분 | 라벨 분포·문장 길이 확인 → TF-IDF + 선형 모델로 기준 점수 |
| 3. Image (35점) | 60분 | 압축 풀기·폴더 구조 확인 → 작은 CNN 으로 기준 점수 |
| 마무리 | 10분 | 제출 파일 6개 이름·열 구조 확인 → 맨 마지막 채점 셀 실행 |

## 시험 자원 가이드

| 구분 | 공식 시험 | 이 모의고사 |
|---|---|---|
| IDE | CPU 8 core · Memory 32G · GPU 1개(VRAM 16GB) | **브라우저 주피터라이트**(설치 없음, CPU) — Colab·로컬 Jupyter 에서도 그대로 실행됨 |
| SW | Python 3.10 · TensorFlow 2.13 · PyTorch 2.1 | Python 3.14(Pyodide) · scikit-learn · LightGBM · XGBoost · Keras 문법(TensorFlow.js) — 위 표의 라이브러리만 |

## 유의 사항

- 막히면 문항 아래 접혀 있는 **힌트 1·2·3단계**를 순서대로 펼쳐 보세요. 단계가 올라갈수록 정답에 가까워집니다.
- 제출 파일명은 **고정**입니다: `answer_1~3` 모델 파일과 `answer_1~3.csv`.


문제에 제시되는 가이드는 필수적으로 준수하여 주시기 바랍니다. 이를 준수하지 않을 경우 채점 제한 또는 0점 처리될 수 있으니
각별히 유의해 주시기 바랍니다.

#### **[학습 정보]**
- 학습 목표 : Tabular · Text · Image 예측 모델 3종을 만들어 달성 목표 이상의 예측 파일을 제출하고, 제출한 코드로 같은 모델이 재현되게 할 수 있다
- AICE 역량 : PT1·PT2 표 데이터 / PX1·PX2 텍스트 / PI1·PI2 이미지 / PS1 제출 파일 형식·재현성
- 제출 파일 : `answer_1~3` 모델 파일(.pkl · .h5 등)과 `answer_1~3.csv` — **파일명은 고정**입니다(이름을 바꾸면 채점되지 않음)

| 문항 | 문제 | 달성 목표 | 역량 | 기술 | 예상 시간 |
|---|---|---|---|---|---|
| 1. Tabular | 탄약 수송 계획의 지연 여부 예측 | F1-score 0.58 이상 | PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | pandas, scikit-learn(ColumnTransformer, Pipeline, OneHotEncoder, StandardScaler, LogisticRegression, StratifiedKFold), LightGBM, joblib | 50분 |
| 2. Text | 스캔한 증명서 문구·명단 대조 값으로 판정결과 분류(4종) | 정확도 0.87 이상 | PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | pandas, scikit-learn(ColumnTransformer, Pipeline, OneHotEncoder, TfidfVectorizer, StratifiedKFold), LightGBM, joblib | 60분 |
| 3. Image | 급식 식판 사진으로 잔반 단계 분류(3종) | 정확도 0.87 이상 | PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | pandas, Keras(TensorFlow.js) | 60분 |

In [1]:
# (브라우저 실습) 주피터라이트에는 tensorflow(TensorFlow.js 대체 구현체)·scikit-learn·LightGBM 등이 미리 설치되어 있어 이 셀은 확인만 합니다.
# Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다.
import os, sys, importlib.util
if 'pyodide' in sys.modules:
    os.environ['AICE_IMG_DIR'] = '/tmp/aice_images/' + os.path.basename(os.getcwd())   # 이미지 압축은 메모리 폴더에 풂(빠름)
    if not ('tensorflow' in sys.modules or importlib.util.find_spec('tensorflow')):
        # 미리 설치되지 않은 사이트(패치 전 주피터라이트)에서만: 같은 폴더의 tfjs_keras.py 를 불러옵니다
        sys.path[:0] = [os.getcwd(), '/drive']
        import tfjs_keras
        await tfjs_keras.load()
print('준비 완료 — 다음 셀로 넘어가세요.')

준비 완료 — 다음 셀로 넘어가세요.


## 1. Tabular 문제 - 탄약 수송 계획의 지연 여부 예측 (30점)

<sub>학습 목표: 탄약 수송 계획의 지연 여부 예측 모델로 테스트 F1-score 0.58 이상을 달성할 수 있다 | 역량: PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | 기술: pandas, scikit-learn(ColumnTransformer, Pipeline, OneHotEncoder, StandardScaler, LogisticRegression, StratifiedKFold), LightGBM, joblib | 난이도: 실전 | 예상 시간: 50분</sub>

### 문제 개요

탄약창에서 부대로 가는 탄약 수송 계획이 **지연될지(1)** 예측하는 이진 분류 문제입니다. 도착지 수용률·제한사유·기상예보·탄약 종류와 함께 **호송조 하나가 맡는 적재량**, **출발 요일** 같은 파생 변수가 중요합니다.

제공된 훈련 데이터셋을 활용해 이진 분류 모델을 개발하고, 테스트 데이터셋의 '지연여부' 값을 예측하시기 바랍니다.

### 달성 목표

"01_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **F1-score 0.58 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `01_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `01_test_x.csv`: 테스트 데이터셋입니다. 라벨('지연여부') 값이 제외되어 있습니다.

**Label 정보**

- '지연여부' : 1 = 지연, 0 = 계획대로 (약 25%)

**Data 정보**

- 'id · 수송코드 · 출발일' : 행 번호 · 수송 번호 · 날짜
- '출발지 · 도착지 · 거리km · 수송수단' : 탄약창A~C · 부대A~E · km · 차량/철도
- '탄약종류 · 수량톤 · 호송조수' : 소화기탄·포탄·폭약류·유도탄 · 톤 · 조
- '기상예보' : 맑음·비·눈·폭우
- '도착지수용률 · 제한사유' : % · 없음·도로제약·기상·수용공간부족
- '계획대비실적차이일' : **학습 데이터에만** 있는 실제 지연 일수 — 정답과 같은 정보라 입력에 쓰면 안 됨

**데이터 출처** — 제7회 데이터한마당 본선 11팀(탄약 물류 AX) 탄약수송 데이터 형식으로 다시 합성(훈련용, 실제 부대 자료 아님) — 수송 3,000건, 거리·수단·기상·호송조 추가

### 데이터 예시

`01_train.csv`

| id | 수송코드 | 출발지 | 도착지 | 출발일 | 거리km | 탄약종류 | 수송수단 | … |
|---|---|---|---|---|---|---|---|---|
| 1 | TRN-02277 | 탄약창B | 부대D | 2024-03-05 | 74 | 포탄 | 철도 | … |
| 2 | TRN-00206 | 탄약창B | 부대D | 2024-07-06 | 67 | 폭약류 | 차량 | … |
| 3 | TRN-02297 | 탄약창A | 부대D | 2025-12-15 | 73 | 소화기탄 | 차량 | … |

`01_test_x.csv`

| id | 수송코드 | 출발지 | 도착지 | 출발일 | 거리km | 탄약종류 | 수송수단 | … |
|---|---|---|---|---|---|---|---|---|
| 1 | TRN-02482 | 탄약창B | 부대C | 2024-02-10 | 68 | 소화기탄 | 차량 | … |
| 2 | TRN-02276 | 탄약창A | 부대C | 2025-06-28 | 42 | 소화기탄 | 철도 | … |
| 3 | TRN-00424 | 탄약창B | 부대B | 2025-12-05 | 299 | 유도탄 | 철도 | … |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 탄약 수송 계획의 지연 여부 예측 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_1.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('지연여부') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_1.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 '지연여부' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 '지연여부' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- `계획대비실적차이일` 은 수송이 끝난 뒤에 정해지는 값(정답 누수)입니다. 테스트에도 없으니 반드시 빼세요.
- 출발일에서 요일을, 수량톤 ÷ 호송조수 로 '조당 적재량' 을 만들어 보세요.
- 평가 지표는 **F1-score(지연=1 기준)** 입니다. 분류 임계값을 교차검증으로 고르세요. 예측값은 0/1 정수로 제출합니다.

### 채점 진행 방식 안내

- 1번 문항 배점 : 30점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_1.csv 로 제출된 예측 데이터를 정답지와 대조하여, **F1-score 0.522 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [2]:

import pandas as pd
import numpy as np
from pathlib import Path
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

# 1. 현재 회차 데이터 로딩
def find_16(names):
    for folder in [Path("."), Path("/mnt/data")]:
        for name in names:
            p = folder / name
            if p.is_file():
                return p
    raise FileNotFoundError(f"파일 없음: {names}")

train_1 = pd.read_csv(find_16([
    "01_train(3).csv", "01_train.csv"
]))
test_1 = pd.read_csv(find_16([
    "01_test_x(4).csv", "01_test_x.csv"
]))

LABEL_1 = "지연여부"

print("훈련:", train_1.shape, "/ 테스트:", test_1.shape)
print("정답 분포:\n", train_1[LABEL_1].value_counts())
print("결측값:", train_1.isna().sum().sum())

# 2. 파생변수 및 전처리
def feats_1(df):
    d = df.drop(columns=[
        "id", "수송코드", "출발일",
        "계획대비실적차이일", LABEL_1
    ], errors="ignore").copy()

    date = pd.to_datetime(df["출발일"])

    d["출발월"] = date.dt.month
    d["출발요일"] = date.dt.dayofweek
    d["주말"] = (date.dt.dayofweek >= 5).astype(int)

    d["조당적재량"] = (
        df["수량톤"] / df["호송조수"].clip(lower=1)
    )
    d["거리x적재량"] = df["거리km"] * d["조당적재량"]
    d["낮은수용률"] = 100 - df["도착지수용률"]

    return pd.get_dummies(d, dtype=int)

X_1 = feats_1(train_1)
y_1 = train_1[LABEL_1]

# 3. 학습·검증 분리
X_tr1, X_val1, y_tr1, y_val1 = train_test_split(
    X_1, y_1,
    test_size=0.2,
    random_state=42,
    stratify=y_1
)

# 4. LightGBM 학습
model_1 = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.04,
    num_leaves=15,
    min_child_samples=25,
    reg_lambda=2.0,
    random_state=42,
    verbosity=-1
)

model_1.fit(X_tr1, y_tr1)

# 5. 확률 예측 및 임계값 조정
prob_1 = model_1.predict_proba(X_val1)[:, 1]

thresholds = np.arange(0.20, 0.61, 0.02)
scores = [
    f1_score(y_val1, (prob_1 >= t).astype(int))
    for t in thresholds
]

best_idx = np.argmax(scores)
threshold_1 = float(thresholds[best_idx])

pred_1 = (prob_1 >= threshold_1).astype(int)
f1_1 = f1_score(y_val1, pred_1)

print(f"\n기본 임계값 F1: {f1_score(y_val1, (prob_1 >= 0.5).astype(int)):.4f}")
print(f"선택 임계값: {threshold_1:.2f}")
print(f"검증 F1: {f1_1:.4f}")
print("달성 목표: F1 >= 0.58")
print("결과:", "목표 달성" if f1_1 >= 0.58 else "추가 개선 필요")
print("\n클래스별 평가:")
print(classification_report(y_val1, pred_1))


훈련: (2250, 15) / 테스트: (750, 13)
정답 분포:
 지연여부
0    1734
1     516
Name: count, dtype: int64
결측값: 0

기본 임계값 F1: 0.6328
선택 임계값: 0.52
검증 F1: 0.6400
달성 목표: F1 >= 0.58
결과: 목표 달성

클래스별 평가:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91       347
           1       0.78      0.54      0.64       103

    accuracy                           0.86       450
   macro avg       0.83      0.75      0.78       450
weighted avg       0.85      0.86      0.85       450



In [3]:

import joblib
import numpy as np
import pandas as pd
from pathlib import Path

# 1. 전체 훈련 데이터 재학습
model_1.fit(X_1, y_1)

# 2. 모델·임계값·컬럼 정보 저장
joblib.dump({
    "model": model_1,
    "threshold": threshold_1,
    "columns": X_1.columns.tolist()
}, "answer_1.pkl")

# 3. 저장 모델 복원
loaded_1 = joblib.load("answer_1.pkl")

# 4. 테스트 전처리
X_test_1 = feats_1(test_1).reindex(
    columns=loaded_1["columns"],
    fill_value=0
)

# 5. 저장 전후 예측 비교
pred_before = (
    model_1.predict_proba(X_test_1)[:, 1] >= threshold_1
).astype(int)

pred_after = (
    loaded_1["model"].predict_proba(X_test_1)[:, 1]
    >= loaded_1["threshold"]
).astype(int)

# 6. 제출 CSV 생성
answer_1 = test_1.copy()
answer_1[LABEL_1] = pred_after
answer_1.to_csv("answer_1.csv", index=False)

# 7. 제출 검증
saved_1 = pd.read_csv("answer_1.csv")

assert Path("answer_1.pkl").is_file()
assert Path("answer_1.csv").is_file()
assert np.array_equal(pred_before, pred_after)
assert len(saved_1) == len(test_1)
assert list(saved_1.columns) == list(test_1.columns) + [LABEL_1]
assert saved_1[LABEL_1].notna().all()
assert set(saved_1[LABEL_1].unique()).issubset({0, 1})
assert saved_1[LABEL_1].tolist() == pred_after.tolist()

pd.testing.assert_frame_equal(
    saved_1[test_1.columns],
    test_1,
    check_dtype=False
)

print("1번 제출 검증 완료")
print("모델: answer_1.pkl")
print("예측: answer_1.csv")
print("제출 형태:", saved_1.shape)
print("분류 임계값:", loaded_1["threshold"])
print("예측 클래스 분포:")
print(saved_1[LABEL_1].value_counts())
print("모델 복원 및 예측값 일치 확인")


1번 제출 검증 완료
모델: answer_1.pkl
예측: answer_1.csv
제출 형태: (750, 14)
분류 임계값: 0.5199999999999998
예측 클래스 분포:
지연여부
0    631
1    119
Name: count, dtype: int64
모델 복원 및 예측값 일치 확인


<details><summary>힌트 1단계 (접근 방향) - 1번 문항</summary>

자료형·결측·양성(1) 비율부터 확인합니다. 번호·날짜 원문·정답과 같은 정보(누수) 열은 빼고, 범주형은 원-핫 인코딩합니다. '참고'의 파생 변수를 만든 뒤 후보 모델 두세 개를 **교차검증 F1** 로 비교하고, 양성이 적으면 **분류 임계값**도 교차검증으로 고르세요.

- 이 문항의 핵심: `계획대비실적차이일` 은 수송이 끝난 뒤에 정해지는 값(정답 누수)입니다. 테스트에도 없으니 반드시 빼세요.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 1번 문항</summary>

사용할 도구: `pandas`, `joblib`, `ColumnTransformer`, `Pipeline`, `OneHotEncoder`, `StandardScaler`, `LogisticRegression`, `LGBMClassifier`, `StratifiedKFold`, `cross_val_predict`, `f1_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 1번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
from sklearn.compose import ____
from sklearn.pipeline import ____
from sklearn.preprocessing import ____, ____
from sklearn.linear_model import ____
from lightgbm import ____
from sklearn.model_selection import ____, ____
from sklearn.metrics import ____

train = pd.read_csv('01_train.csv')
test = pd.read_csv('01_test_x.csv')
LABEL = '____'
print(train.shape, test.shape, '____', round(train[LABEL].mean(), ____))

def feats(df):
    d = df.copy()
    d['____'] = d['____'] / d['____']                        # 호송조 하나가 맡는 톤수
    d['____'] = (d['____'] - ____).clip(lower=____)
    w = pd.to_datetime(d['____']).dt.weekday
    d['____'] = (w >= ____).astype(int)
    d['____'] = pd.to_datetime(d['____']).dt.month
    return d

# ▷ 후보 모델을 교차검증으로 비교(점수 출력)
# ▷ 분류 임계값을 교차검증으로 고르기
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_1.pkl`(모델) · `answer_1.csv`(예측)

</details>

## 2. Text 문제 - 스캔한 증명서 문구·명단 대조 값으로 판정결과 분류(4종) (35점)

<sub>학습 목표: 스캔한 증명서 문구·명단 대조 값으로 판정결과 분류(4종) 모델로 테스트 정확도 0.87 이상을 달성할 수 있다 | 역량: PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | 기술: pandas, scikit-learn(ColumnTransformer, Pipeline, OneHotEncoder, TfidfVectorizer, StratifiedKFold), LightGBM, joblib | 난이도: 중급 | 예상 시간: 60분</sub>

### 문제 개요

교육 수료·표창 증명서를 **확인완료 · 확인필요(오탈자·직인 불명확) · 명단없음 · 기간불일치** 4종으로 판정합니다. 명단 대조 값(이름 유사도·기간 차이)과 스캔한 **증명서 문구(텍스트)** 의 오탈자를 함께 봐야 합니다.

제공된 훈련 데이터셋을 활용해 텍스트 다중 분류(4종) 모델을 개발하고, 테스트 데이터셋의 '판정결과' 값을 예측하시기 바랍니다.

### 달성 목표

"02_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 87% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `02_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `02_test_x.csv`: 테스트 데이터셋입니다. 라벨('판정결과') 값이 제외되어 있습니다.

**Label 정보**

- '판정결과' : 확인완료 / 확인필요 / 명단없음 / 기간불일치

**Data 정보**

- 'id' : 행 번호
- '교육유형' : 의무교육·포상·자격교육·안전교육
- '증명서문구' : 스캔해 읽은 증명서 글(오탈자가 있을 수 있음)
- '명단이름유사도 · 기간차이일' : 교육 명단과 이름 유사도(0~1) · 기간 차이(일)
- '발급기관일치 · 스캔품질 · 처리시간초' : 1/0 · 0~1 · 초

**데이터 출처** — 제7회 데이터한마당 본선 22팀(증명서 자동검증) 증명서검증 데이터 형식으로 다시 합성(훈련용, 실제 부대 자료 아님) — 증명서 3,200건

### 데이터 예시

`02_train.csv`

| id | 교육유형 | 증명서문구 | 명단이름유사도 | 기간차이일 | 발급기관일치 | 스캔품질 | 처리시간초 | … |
|---|---|---|---|---|---|---|---|---|
| 1 | 안전교육 | 성명 최예준 위 사람은 국방대학교에서 실시한 산업 안전 교육을(를) 2024.10.14부터 2024.10.24까지 교육 기간 동안 성실히 이수하였음을 증명함 — 수료 증명서 발급 국방대학교 | 0.977 | 17 | 1 | 0.79 | 5.1 | … |
| 2 | 자격교육 | 성명 한연우 위 사람은 육군종합정비창에서 실시한 운전 교육 과정을(를) 2025.02.02부터 2025.02.07까지 교육 기간 동안 성실히 이수하였음을 증명함 — 수로 증명서 발급 육군종합정비창 | 0.928 | -1 | 0 | 0.68 | 6.9 | … |
| 3 | 포상 | 성명 오서진 위 사람은 육군종합정비창에서 실시한 모범 장병 표창을(를) 2024.11.26부터 2024.12.06까지 이수하였기에 이 상장을 수여함 — 표창 증명서 발급 육군종합정비창 | 0.902 | 1 | 1 | 0.76 | 4.3 | … |

`02_test_x.csv`

| id | 교육유형 | 증명서문구 | 명단이름유사도 | 기간차이일 | 발급기관일치 | 스캔품질 | 처리시간초 |
|---|---|---|---|---|---|---|---|
| 1 | 의무교육 | 성명 서서진 위 사람은 국방대학교에서 실시한 청렴 교육을(를) 2024.10.15부터 2024.10.29까지 교육 기간 동안 성실히 이수하였음을 증명함 — 수료 증명서 발급 국방대학교 | 0.262 | 0 | 1 | 0.66 | 11 |
| 2 | 자격교육 | 성명 윤민준 위 사람은 육군종합정비창에서 실시한 운전 교육 과정을(를) 2025.07.07부터 2025.07.10까지 교육 기간 동안 성실히 이수하였음을 증명함 — 수료 증명서 발급 육군종합정비창 | 0.872 | 0 | 1 | 0.7 | 1.6 |
| 3 | 의무교육 | 성명 윤지호 위 사람은 국방전산정보원에서 실시한 청렴 교육을(를) 2025.07.11부터 2025.07.12까지 교육 기간 동안 성실히 이수하였음을 증명함 — 수료 증명서 발급 국방전산정보원 | 0.243 | 0 | 1 | 0.95 | 5.3 |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 스캔한 증명서 문구·명단 대조 값으로 판정결과 분류(4종) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_2.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('판정결과') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_2.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 '판정결과' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 '판정결과' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 오탈자는 '교윢'·'수료ㅇ' 처럼 글자 단위로 나타나 **글자 n-gram** 이 잘 잡습니다. 낱자(ㄱ~ㅎ, ㅏ~ㅣ) 개수 같은 파생 변수도 도움이 됩니다.
- 숫자(이름 유사도·기간 차이·스캔 품질)는 명단없음·기간불일치·직인 불명확을 가르는 핵심입니다. 텍스트와 함께 넣으세요.
- 평가 지표는 **정확도(Accuracy)** 입니다.

### 채점 진행 방식 안내

- 2번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_2.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 78.3% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [5]:

import pandas as pd
from pathlib import Path
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

# 1. 데이터 로딩
def find_16_2(names):
    for folder in [Path("."), Path("/mnt/data")]:
        for name in names:
            p = folder / name
            if p.is_file():
                return p
    raise FileNotFoundError(names)

train_2 = pd.read_csv(find_16_2([
    "02_train(3).csv", "02_train.csv"
]))
test_2 = pd.read_csv(find_16_2([
    "02_test_x(3).csv", "02_test_x.csv"
]))

LABEL_2 = "판정결과"

print("훈련:", train_2.shape, "/ 테스트:", test_2.shape)
print("정답 분포:\n", train_2[LABEL_2].value_counts())
print("결측값:", train_2.isna().sum().sum())

# 2. 특징 생성
def feats_2(df):
    d = df.drop(columns=["id", LABEL_2], errors="ignore").copy()
    d["기간차이절대값"] = d["기간차이일"].abs()
    d["문구길이"] = d["증명서문구"].str.len()
    d["한글낱자수"] = d["증명서문구"].str.count(r"[ㄱ-ㅎㅏ-ㅣ]")
    return d

X_2 = feats_2(train_2)
y_2 = train_2[LABEL_2]

numeric_cols_2 = [
    "명단이름유사도", "기간차이일", "기간차이절대값",
    "발급기관일치", "스캔품질", "처리시간초",
    "문구길이", "한글낱자수"
]

# 3. 전처리 및 모델
preprocessor_2 = ColumnTransformer([
    ("text", TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 4),
        min_df=3,
        max_features=3500,
        sublinear_tf=True
    ), "증명서문구"),

    ("num", StandardScaler(), numeric_cols_2),

    ("cat", OneHotEncoder(
        handle_unknown="ignore"
    ), ["교육유형"])
])

model_2 = Pipeline([
    ("preprocess", preprocessor_2),
    ("classifier", LGBMClassifier(
        n_estimators=220,
        learning_rate=0.05,
        num_leaves=15,
        random_state=42,
        verbosity=-1
    ))
])

# 4. 학습·검증 분리
X_tr2, X_val2, y_tr2, y_val2 = train_test_split(
    X_2, y_2,
    test_size=0.2,
    random_state=42,
    stratify=y_2
)

# 5. 학습 및 성능 검증
model_2.fit(X_tr2, y_tr2)
pred_2 = model_2.predict(X_val2)
acc_2 = accuracy_score(y_val2, pred_2)

print(f"\n검증 Accuracy: {acc_2:.4%}")
print("달성 목표: 87%")
print("결과:", "목표 달성" if acc_2 >= 0.87 else "추가 개선 필요")
print("\n클래스별 평가:")
print(classification_report(y_val2, pred_2))


훈련: (2560, 9) / 테스트: (640, 8)
정답 분포:
 판정결과
확인완료     1265
확인필요      534
기간불일치     390
명단없음      371
Name: count, dtype: int64
결측값: 0

검증 Accuracy: 93.3594%
달성 목표: 87%
결과: 목표 달성

클래스별 평가:
              precision    recall  f1-score   support

       기간불일치       0.99      0.95      0.97        78
        명단없음       1.00      0.92      0.96        74
        확인완료       0.90      0.98      0.94       253
        확인필요       0.95      0.81      0.87       107

    accuracy                           0.93       512
   macro avg       0.96      0.92      0.93       512
weighted avg       0.94      0.93      0.93       512



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:

import joblib
import numpy as np
import pandas as pd
from pathlib import Path

# 1. 전체 훈련 데이터 재학습
model_2.fit(X_2, y_2)

# 2. 모델 저장 및 복원
joblib.dump(model_2, "answer_2.pkl")
loaded_2 = joblib.load("answer_2.pkl")

# 3. 테스트 데이터 전처리
X_test_2 = feats_2(test_2)
X_test_2 = X_test_2[X_2.columns]

# 4. 저장 전후 예측 비교
pred_before_2 = model_2.predict(X_test_2)
pred_after_2 = loaded_2.predict(X_test_2)

# 5. 제출 CSV 생성
answer_2 = test_2.copy()
answer_2[LABEL_2] = pred_after_2
answer_2.to_csv("answer_2.csv", index=False)

# 6. 제출 파일 검증
saved_2 = pd.read_csv("answer_2.csv")

assert Path("answer_2.pkl").is_file()
assert Path("answer_2.csv").is_file()
assert np.array_equal(pred_before_2, pred_after_2)

assert len(saved_2) == len(test_2)
assert list(saved_2.columns) == list(test_2.columns) + [LABEL_2]
assert saved_2[LABEL_2].notna().all()
assert set(saved_2[LABEL_2]).issubset(set(y_2.unique()))
assert saved_2[LABEL_2].tolist() == pred_after_2.tolist()

pd.testing.assert_frame_equal(
    saved_2[test_2.columns],
    test_2,
    check_dtype=False
)

print("2번 제출 검증 완료")
print("모델: answer_2.pkl")
print("예측: answer_2.csv")
print("제출 형태:", saved_2.shape)
print("\n예측 클래스 분포:")
print(saved_2[LABEL_2].value_counts())
print("\n모델 복원 및 예측값 일치 확인")


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


2번 제출 검증 완료
모델: answer_2.pkl
예측: answer_2.csv
제출 형태: (640, 9)

예측 클래스 분포:
판정결과
확인완료     341
확인필요     115
기간불일치     98
명단없음      86
Name: count, dtype: int64

모델 복원 및 예측값 일치 확인


<details><summary>힌트 1단계 (접근 방향) - 2번 문항</summary>

라벨 분포와 문장 길이를 확인합니다. 문장을 TF-IDF 로 바꾸되 **단어 단위와 글자 조각(char n-gram)** 을 비교하고, 나이브 베이즈·선형 SVM·로지스틱 회귀 중 **교차검증 정확도** 가 높은 조합으로 전체 학습하세요. 다른 컬럼(숫자·범주)이 단서이면 `ColumnTransformer` 로 함께 넣습니다.

- 이 문항의 핵심: 오탈자는 '교윢'·'수료ㅇ' 처럼 글자 단위로 나타나 **글자 n-gram** 이 잘 잡습니다. 낱자(ㄱ~ㅎ, ㅏ~ㅣ) 개수 같은 파생 변수도 도움이 됩니다.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 2번 문항</summary>

사용할 도구: `pandas`, `joblib`, `ColumnTransformer`, `Pipeline`, `OneHotEncoder`, `TfidfVectorizer`, `LGBMClassifier`, `StratifiedKFold`, `cross_val_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html>
- 참고 문서: <https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 2번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
from sklearn.compose import ____
from sklearn.pipeline import ____
from sklearn.preprocessing import ____
from sklearn.feature_extraction.text import ____
from lightgbm import ____
from sklearn.model_selection import ____, ____

train = pd.read_csv('02_train.csv')
test = pd.read_csv('02_test_x.csv')

# ▷ 후보 모델을 교차검증으로 비교(점수 출력)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_2.pkl`(모델) · `answer_2.csv`(예측)

</details>

## 3. Image 문제 - 급식 식판 사진으로 잔반 단계 분류(3종) (35점)

<sub>학습 목표: 급식 식판 사진으로 잔반 단계 분류(3종) 모델로 테스트 정확도 0.87 이상을 달성할 수 있다 | 역량: PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | 기술: pandas, Keras(TensorFlow.js) | 난이도: 실전 | 예상 시간: 60분</sub>

### 문제 개요

배식 후 반납된 **식판 사진**으로 잔반 단계(적음·보통·많음)를 분류해 메뉴별 잔반량을 자동 집계하는 문제입니다. 반찬 색·위치·수저가 사진마다 달라 '얼마나 덮여 있는지'를 잡아야 합니다.

제공된 훈련 데이터셋을 활용해 이미지 다중 분류(3종) 모델을 개발하고, 테스트 데이터셋의 'label' 값을 예측하시기 바랍니다.

### 달성 목표

"03_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 87% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `03_images.zip`: 압축을 풀면 `03_train/<클래스>/tray_*.png`(학습)와 `03_test/test_*.png`(테스트) 폴더가 생깁니다. 64×64 컬러 PNG.
- `03_train.csv`: 학습 이미지 파일명과 라벨
- `03_test_x.csv`: 테스트 이미지 파일명(라벨 제외)

**Label 정보**

- 'label' : little: 잔반 적음 / medium: 잔반 보통 / much: 잔반 많음

**데이터 출처** — 제7회 데이터한마당 본선 24팀(급식이음) 주제로 합성한 식판 사진(훈련용) — 3단계 × 400장

### 데이터 예시

학습 이미지 라벨별 장수: little 320, much 320, medium 320

| file_name | label |
|---|---|
| tray_00001.png | little |
| tray_00002.png | little |
| tray_00003.png | much |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 급식 식판 사진으로 잔반 단계 분류(3종) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_3.h5** (또는 다른 확장자의 모델 파일)

(2) 라벨('label') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_3.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'label' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'label' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 식판은 좌우를 뒤집어도 잔반 양이 같으니 좌우 뒤집기 증강을 써 보세요.
- 작은 CNN(`Sequential` + `Conv2D`)을 직접 학습합니다. 1/255 로 나눠 0~1 로 맞추고, Dropout·조기 종료로 과적합을 막으세요.
- 평가 지표는 **정확도(Accuracy)** 입니다. 테스트는 `03_test_x.csv` 의 파일명 순서대로 읽어 예측하세요.

### 채점 진행 방식 안내

- 3번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_3.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 78.3% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [8]:

import os, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Flatten, Dense, Dropout
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. 16회차 파일 불러오기
def find_16_3(names):
    for folder in [Path("."), Path("/mnt/data")]:
        for name in names:
            p = folder / name
            if p.is_file():
                return p
    raise FileNotFoundError(names)

train_3 = pd.read_csv(find_16_3([
    "03_train(3).csv", "03_train.csv"
]))
test_3 = pd.read_csv(find_16_3([
    "03_test_x(3).csv", "03_test_x.csv"
]))

zip_path = find_16_3([
    "03_images(3).zip", "03_images.zip"
])

# 2. 이미지 압축 해제
IMG_DIR_3 = Path("aice16_images")

if not (IMG_DIR_3 / "03_train").is_dir():
    IMG_DIR_3.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(IMG_DIR_3)

# 3. 이미지 로딩 및 정규화
SIZE_3 = 64
CLASSES_3 = sorted(train_3["label"].unique())
label_map_3 = {v: i for i, v in enumerate(CLASSES_3)}

def load_image_3(path):
    with Image.open(path) as im:
        return np.asarray(
            im.convert("RGB").resize((SIZE_3, SIZE_3)),
            dtype=np.float32
        ) / 255.0

X_3 = np.stack([
    load_image_3(IMG_DIR_3 / "03_train" / label / filename)
    for filename, label in zip(
        train_3["file_name"], train_3["label"]
    )
])

y_3 = train_3["label"].map(label_map_3).to_numpy()

print("훈련:", X_3.shape, "/ 테스트:", test_3.shape)
print("클래스:", CLASSES_3)
print("정답 분포:\n", train_3["label"].value_counts())
print("결측값:", train_3.isna().sum().sum())

# 4. 학습·검증 분리
X_tr3, X_val3, y_tr3, y_val3 = train_test_split(
    X_3, y_3,
    test_size=0.2,
    random_state=42,
    stratify=y_3
)

# 5. 데이터 증강
augment_3 = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.08,
    fill_mode="nearest"
)

# 6. CNN 모델
np.random.seed(42)
tf.random.set_seed(42)

model_3 = Sequential([
    Input(shape=(64, 64, 3)),

    Conv2D(32, 3, activation="relu", padding="same"),
    Conv2D(32, 3, activation="relu", padding="same"),
    MaxPooling2D(),
    BatchNormalization(),

    Conv2D(64, 3, activation="relu", padding="same"),
    MaxPooling2D(),
    BatchNormalization(),

    Conv2D(128, 3, activation="relu", padding="same"),
    MaxPooling2D(),
    BatchNormalization(),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.4),
    Dense(len(CLASSES_3), activation="softmax")
])

model_3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 7. 콜백 설정
BEST_MODEL_3 = "aice16_best_3.keras"

callbacks_3 = [
    EarlyStopping(
        monitor="val_accuracy",
        patience=7,
        mode="max",
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    ),
    ModelCheckpoint(
        BEST_MODEL_3,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True
    )
]

# 8. 모델 학습
history_3 = model_3.fit(
    augment_3.flow(
        X_tr3, y_tr3,
        batch_size=32,
        seed=42
    ),
    validation_data=(X_val3, y_val3),
    epochs=30,
    callbacks=callbacks_3,
    verbose=1
)

# 9. 최고 성능 모델 복원 및 검증
from tensorflow.keras.models import load_model

model_3 = load_model(BEST_MODEL_3)

pred_3 = np.argmax(
    model_3.predict(X_val3, verbose=0),
    axis=1
)

acc_3 = accuracy_score(y_val3, pred_3)

print(f"\n검증 Accuracy: {acc_3:.4%}")
print("달성 목표: 87%")
print("결과:", "목표 달성" if acc_3 >= 0.87 else "추가 개선 필요")
print("\n클래스별 평가:")
print(classification_report(
    y_val3, pred_3,
    target_names=CLASSES_3,
    zero_division=0
))


훈련: (960, 64, 64, 3) / 테스트: (240, 1)
클래스: ['little', 'medium', 'much']
정답 분포:
 label
little    320
much      320
medium    320
Name: count, dtype: int64
결측값: 0
Epoch 1/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 22s 804ms/step - accuracy: 0.6510 - loss: 1.5584 - val_accuracy: 0.3333 - val_loss: 2.4674 - learning_rate: 0.0010
Epoch 2/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 21s 900ms/step - accuracy: 0.7852 - loss: 0.6231 - val_accuracy: 0.3333 - val_loss: 5.7933 - learning_rate: 0.0010
Epoch 3/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 19s 783ms/step - accuracy: 0.8320 - loss: 0.3878 - val_accuracy: 0.3333 - val_loss: 9.4320 - learning_rate: 0.0010
Epoch 4/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 19s 777ms/step - accuracy: 0.8281 - loss: 0.4182 - val_accuracy: 0.3333 - val_loss: 10.6836 - learning_rate: 0.0010
Epoch 5/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 21s 770ms/step - accuracy: 0.8789 - loss: 0.3055 - val_accuracy: 0.3333 - val_loss: 12.8465 - learning_rate: 5.0000e-04
Epoch 6/30
24/24 ━━━━━━━━━━━━━━━━━━━━ 19s 776ms/step - accuracy: 0.88

In [9]:

import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# 1. HOG 특징 추출 (numpy 구현)
def hog_features_3(gray):
    cell, bins = 8, 9
    gx = np.zeros_like(gray)
    gy = np.zeros_like(gray)

    gx[:, 1:-1] = gray[:, 2:] - gray[:, :-2]
    gy[1:-1, :] = gray[2:, :] - gray[:-2, :]

    magnitude = np.hypot(gx, gy)
    angle = (np.rad2deg(np.arctan2(gy, gx)) % 180) / 20

    hist = np.zeros((8, 8, bins), dtype=np.float32)

    for i in range(8):
        for j in range(8):
            a = angle[i*cell:(i+1)*cell,
                      j*cell:(j+1)*cell].ravel()
            m = magnitude[i*cell:(i+1)*cell,
                          j*cell:(j+1)*cell].ravel()

            low = np.floor(a).astype(int) % bins
            frac = a - np.floor(a)

            hist[i, j] += np.bincount(
                low, weights=m*(1-frac), minlength=bins
            )
            hist[i, j] += np.bincount(
                (low+1) % bins,
                weights=m*frac, minlength=bins
            )

    blocks = []
    for i in range(7):
        for j in range(7):
            b = hist[i:i+2, j:j+2].ravel().copy()
            b /= np.sqrt(np.sum(b*b) + 1e-6)
            b = np.minimum(b, 0.2)
            b /= np.sqrt(np.sum(b*b) + 1e-6)
            blocks.extend(b)

    return np.asarray(blocks, dtype=np.float32)

# 2. 색상 및 질감 특징
def extract_features_3(image):
    rgb = np.rint(image * 255).clip(0, 255).astype(np.uint8)
    hsv = np.asarray(
        Image.fromarray(rgb).convert("HSV"),
        dtype=np.float32
    ) / 255.0

    color_features = []

    for n in [1, 2, 4, 8]:
        for i in range(n):
            for j in range(n):
                tile = hsv[
                    i*64//n:(i+1)*64//n,
                    j*64//n:(j+1)*64//n
                ]

                color_features.extend(tile.mean(axis=(0, 1)))
                color_features.extend(tile.std(axis=(0, 1)))

                for ch, bins in [
                    (0, 12 if n <= 2 else 4),
                    (1, 10 if n <= 2 else 4),
                    (2, 10 if n <= 2 else 4)
                ]:
                    counts = np.histogram(
                        tile[:, :, ch],
                        bins=bins,
                        range=(0, 1)
                    )[0]
                    color_features.extend(
                        counts / (tile.shape[0] * tile.shape[1])
                    )

    gray = (
        0.2125*rgb[:, :, 0] +
        0.7154*rgb[:, :, 1] +
        0.0721*rgb[:, :, 2]
    ).astype(np.float32) / 255.0

    return np.concatenate([
        np.asarray(color_features, dtype=np.float32),
        hog_features_3(gray)
    ])

# 3. 기존 이미지 데이터에서 특징 추출
features_3 = np.stack([
    extract_features_3(img) for img in X_3
])

print("특징 데이터:", features_3.shape)

# 4. 동일한 검증 분할
F_tr3, F_val3, y_tr3, y_val3 = train_test_split(
    features_3, y_3,
    test_size=0.2,
    random_state=42,
    stratify=y_3
)

# 5. 색상과 HOG 특징 개별 표준화
COLOR_DIM_3 = 1630

preprocessor_svc_3 = ColumnTransformer([
    ("color", StandardScaler(),
     list(range(COLOR_DIM_3))),
    ("hog", StandardScaler(),
     list(range(COLOR_DIM_3, features_3.shape[1])))
], transformer_weights={"hog": 0.5})

model_svc_3 = Pipeline([
    ("preprocess", preprocessor_svc_3),
    ("svc", SVC(
        kernel="rbf",
        C=10,
        gamma="scale"
    ))
])

# 6. 학습 및 검증
model_svc_3.fit(F_tr3, y_tr3)

pred_svc_3 = model_svc_3.predict(F_val3)
acc_svc_3 = accuracy_score(y_val3, pred_svc_3)

print(f"\n검증 Accuracy: {acc_svc_3:.4%}")
print("달성 목표: 87%")
print("결과:", "목표 달성" if acc_svc_3 >= 0.87 else "추가 개선 필요")
print("\n클래스별 평가:")
print(classification_report(
    y_val3, pred_svc_3,
    target_names=CLASSES_3,
    zero_division=0
))


특징 데이터: (960, 3394)

검증 Accuracy: 88.5417%
달성 목표: 87%
결과: 목표 달성

클래스별 평가:
              precision    recall  f1-score   support

      little       0.90      0.98      0.94        64
      medium       0.89      0.75      0.81        64
        much       0.87      0.92      0.89        64

    accuracy                           0.89       192
   macro avg       0.89      0.89      0.88       192
weighted avg       0.89      0.89      0.88       192



In [10]:

import joblib
import numpy as np
import pandas as pd
from pathlib import Path

# 1. 전체 훈련 데이터 재학습
model_svc_3.fit(features_3, y_3)

# 2. 모델 및 클래스 정보 저장
joblib.dump({
    "model": model_svc_3,
    "classes": CLASSES_3,
    "feature_dim": features_3.shape[1],
    "image_size": SIZE_3
}, "answer_3.pkl")

# 3. 저장 모델 복원
loaded_3 = joblib.load("answer_3.pkl")

# 4. 테스트 이미지 로딩 및 특징 추출
X_test_3 = np.stack([
    load_image_3(IMG_DIR_3 / "03_test" / fn)
    for fn in test_3["file_name"]
])

features_test_3 = np.stack([
    extract_features_3(img) for img in X_test_3
])

assert features_test_3.shape[1] == loaded_3["feature_dim"]

# 5. 저장 전후 예측 비교
pred_before_3 = model_svc_3.predict(features_test_3)
pred_after_3 = loaded_3["model"].predict(features_test_3)

assert np.array_equal(pred_before_3, pred_after_3)

# 6. 원래 클래스 이름으로 변환
pred_labels_3 = np.array(
    loaded_3["classes"]
)[pred_after_3]

# 7. 제출 CSV 생성
answer_3 = test_3.copy()
answer_3["label"] = pred_labels_3
answer_3.to_csv("answer_3.csv", index=False)

# 8. 제출 파일 검증
saved_3 = pd.read_csv("answer_3.csv")

assert Path("answer_3.pkl").is_file()
assert Path("answer_3.csv").is_file()

assert len(saved_3) == len(test_3)
assert list(saved_3.columns) == list(test_3.columns) + ["label"]
assert saved_3["label"].notna().all()
assert set(saved_3["label"]).issubset(set(CLASSES_3))
assert saved_3["label"].tolist() == pred_labels_3.tolist()

pd.testing.assert_frame_equal(
    saved_3[test_3.columns],
    test_3,
    check_dtype=False
)

# 9. 결과 출력
print("3번 제출 검증 완료")
print("모델: answer_3.pkl")
print("예측: answer_3.csv")
print("제출 형태:", saved_3.shape)
print("\n예측 클래스 분포:")
print(saved_3["label"].value_counts())
print("\n모델 복원 및 예측값 일치 확인")


3번 제출 검증 완료
모델: answer_3.pkl
예측: answer_3.csv
제출 형태: (240, 2)

예측 클래스 분포:
label
much      81
medium    81
little    78
Name: count, dtype: int64

모델 복원 및 예측값 일치 확인


<details><summary>힌트 1단계 (접근 방향) - 3번 문항</summary>

압축을 풀고 폴더 구조(학습: 클래스별 폴더, 테스트: 한 폴더)를 확인합니다. 이미지를 같은 크기로 읽어 0~1 로 맞추고, 검증 데이터를 떼어 **작은 CNN**(또는 사전학습 모델 특징 + 분류기)을 학습합니다. 증강·Dropout·조기 종료로 과적합을 막고, 테스트는 `03_test_x.csv` 순서대로 예측하세요.

- 이 문항의 핵심: 식판은 좌우를 뒤집어도 잔반 양이 같으니 좌우 뒤집기 증강을 써 보세요.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 3번 문항</summary>

사용할 도구: `tensorflow`, `pandas`, `train_test_split`, `load_img`, `img_to_array`, `Sequential`, `load_model`, `Input`, `Conv2D`, `MaxPooling2D`, `Flatten`, `Dense`, `Dropout`, `EarlyStopping`.

- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html>
- 참고 문서: <https://keras.io/api/data_loading/image/#loadimg-function>
- 참고 문서: <https://keras.io/api/data_loading/image/#imgtoarray-function>
- 참고 문서: <https://keras.io/api/models/sequential/>
- 참고 문서: <https://keras.io/api/layers/convolution_layers/convolution2d/>
- 참고 문서: <https://keras.io/api/layers/pooling_layers/max_pooling2d/>

</details>

<details><summary>힌트 3단계 (코드 구조) - 3번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
IMG_DIR = os.environ.get('____', '____')        # 압축을 풀 위치(브라우저판은 메모리 폴더 /tmp)
if not os.path.exists(os.path.join(IMG_DIR, '____')):
    zipfile.ZipFile('03_images.zip').extractall(IMG_DIR)
import ____
from sklearn.model_selection import ____
from tensorflow.keras.utils import ____, ____
from tensorflow.keras.models import ____, ____
from tensorflow.keras.layers import ____, ____, ____, ____, ____, ____
from tensorflow.keras.callbacks import ____, ____, ____

train = pd.read_csv('03_train.csv')
test = pd.read_csv('03_test_x.csv')
SIZE = ____                                                 # 브라우저 메모리·속도를 위해 ____×____ 로 읽음
CLASSES = sorted(train['____'].unique())                 # 라벨 번호 = 알파벳 순
X = np.stack([img_to_array(load_img(f'____', target_size=(SIZE, SIZE)))
              for f, l in zip(train['____'], train['____'])]) / ____

# ▷ 모델 구성(CNN 또는 사전학습 특징 추출)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_3.h5`(모델) · `answer_3.csv`(예측)

</details>